# 01 – Tái Hiện Kết Quả Chính (Main Experiments)

**Bài báo:** Hyperspherical Prototype Node Clustering (HPNC)  
**Nguồn:** Transactions on Machine Learning Research, 01/2024  
**Môn học:** Khai thác dữ liệu và ứng dụng – CSC14004

---

## Mục tiêu

Tái hiện **Table 1** trong bài báo: so sánh hiệu năng phân cụm (ACC, NMI, ARI)  
của **HPNC-IM** và **HPNC-DEC** trên 5 benchmark datasets:  
`Cora`, `CiteSeer`, `PubMed`, `ACM`, `DBLP`

Kết quả được tổng hợp vào bảng so sánh với số liệu gốc từ bài báo và tính **độ lệch tương đối** cho từng chỉ số.

---

## Pipeline tổng quát của HPNC (từ bài báo)

```
Input graph (X, A)
      │
      ▼
[Masked Graph Autoencoder]   ← Eq. 3 (L_fea) + Eq. 5 (L_edge)
   GATEncoder (2 layers, 4 heads × 128d, PReLU, dropout)
      │ Z  (n × m)
      ▼
[L2-Normalize]  →  Z̃  trên unit-hypersphere
      │
      ▼
[Rotated Clustering Affinity]  ←  Eq. 6
   Q_{i,j} = softmax( Z̃_i · R·μ̃_j )   (R: learnable orthogonal matrix)
      │ Q  (n × c)
      ▼
[Clustering Loss]
   HPNC-IM : L = L_fea + α·L_edge − β·L_bal + γ·L_ent   (Eq. 10)
   HPNC-DEC: L = L_fea + α·L_edge − β·L_bal + γ·L_DEC   (Eq. 14)
      │
      ▼
Predicted labels = argmax(Q)   (không cần K-means)
```


## 1. Import thư viện và cấu hình chung

In [ ]:
import sys, os, time, warnings, json
warnings.filterwarnings('ignore')

# Thêm src/ vào Python path để import model, metrics, utils
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
sys.path.insert(0, SRC_DIR)

import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['figure.dpi'] = 120

from torch_geometric.datasets import Planetoid, AttributedGraphDataset
import torch_geometric.transforms as T
from sklearn.cluster import KMeans

from model import HPNC_IM, HPNC_DEC
from metrics import evaluate_clustering

# ── Seed cố định cho reproducibility ──────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")
print(f"SRC dir : {SRC_DIR}")


## 2. Số liệu gốc từ bài báo (Table 1)

Đây là kết quả được báo cáo trong bài báo (trung bình ± độ lệch chuẩn qua 5 lần chạy).


In [ ]:
# ── Số liệu từ Table 1 trong bài báo ─────────────────────────────────────
# Format: {dataset: {scheme: {metric: (mean, std)}}}
PAPER_RESULTS = {
    'Cora': {
        'HPNC-IM':  {'ACC': (74.1, 0.5), 'NMI': (58.9, 0.6), 'ARI': (51.5, 0.9)},
        'HPNC-DEC': {'ACC': (74.4, 0.5), 'NMI': (59.4, 0.9), 'ARI': (51.9, 1.2)},
    },
    'CiteSeer': {
        'HPNC-IM':  {'ACC': (72.4, 0.5), 'NMI': (46.3, 0.7), 'ARI': (48.4, 0.7)},
        'HPNC-DEC': {'ACC': (73.0, 0.5), 'NMI': (46.9, 0.7), 'ARI': (49.7, 0.7)},
    },
    'PubMed': {
        'HPNC-IM':  {'ACC': (70.4, 0.4), 'NMI': (34.6, 0.7), 'ARI': (33.5, 0.7)},
        'HPNC-DEC': {'ACC': (70.6, 0.9), 'NMI': (33.1, 1.5), 'ARI': (33.3, 1.4)},
    },
    'ACM': {
        'HPNC-IM':  {'ACC': (92.1, 0.1), 'NMI': (72.1, 0.3), 'ARI': (78.1, 0.3)},
        'HPNC-DEC': {'ACC': (92.3, 0.1), 'NMI': (72.4, 0.3), 'ARI': (78.4, 0.2)},
    },
    'DBLP': {
        'HPNC-IM':  {'ACC': (80.0, 0.4), 'NMI': (49.8, 0.4), 'ARI': (54.9, 0.6)},
        'HPNC-DEC': {'ACC': (80.1, 0.5), 'NMI': (49.9, 0.5), 'ARI': (55.0, 0.7)},
    },
}

# Hiển thị bảng
rows = []
for ds in PAPER_RESULTS:
    for scheme in PAPER_RESULTS[ds]:
        r = PAPER_RESULTS[ds][scheme]
        rows.append({
            'Dataset': ds, 'Scheme': scheme,
            'ACC': f"{r['ACC'][0]:.1f} ± {r['ACC'][1]:.1f}",
            'NMI': f"{r['NMI'][0]:.1f} ± {r['NMI'][1]:.1f}",
            'ARI': f"{r['ARI'][0]:.1f} ± {r['ARI'][1]:.1f}",
        })

df_paper = pd.DataFrame(rows)
print("=== Số liệu gốc – Table 1 (bài báo) ===")
print(df_paper.to_string(index=False))


## 3. Tải và tiền xử lý dữ liệu

Dữ liệu được tải tự động qua `torch_geometric`:
- **Cora, CiteSeer, PubMed**: `Planetoid` dataset
- **ACM, DBLP**: `AttributedGraphDataset`

Theo **Table 3** trong bài báo (Appendix A.1).


In [ ]:
def load_data(name: str, data_root: str = None) -> dict:
    """
    Tải dataset đồ thị và chuyển sang format dùng cho HPNC.

    Parameters
    ----------
    name : str
        Tên dataset: 'Cora', 'CiteSeer', 'PubMed', 'ACM', 'DBLP'.
    data_root : str, optional
        Thư mục lưu cache. Mặc định = ../data.

    Returns
    -------
    dict
        x, edge_index, y, num_nodes, num_features, num_classes, num_edges, name.
    """
    if data_root is None:
        data_root = os.path.join(PROJECT_ROOT, 'data')
    os.makedirs(data_root, exist_ok=True)

    if name.lower() in ['cora', 'citeseer', 'pubmed']:
        dataset = Planetoid(
            root=data_root,
            name=name,
            transform=T.NormalizeFeatures()
        )
        data = dataset[0]
    elif name.lower() in ['acm', 'dblp']:
        dataset = AttributedGraphDataset(
            root=data_root,
            name=name,
            transform=T.NormalizeFeatures()
        )
        data = dataset[0]
    else:
        raise ValueError(f"Dataset '{name}' không được hỗ trợ.")

    return {
        'x':            data.x.to(DEVICE),
        'edge_index':   data.edge_index.to(DEVICE),
        'y':            data.y.cpu().numpy(),
        'num_nodes':    data.num_nodes,
        'num_features': data.num_features,
        'num_classes':  int(data.y.max().item()) + 1,
        'num_edges':    data.edge_index.shape[1],
        'name':         name,
    }


# Thống kê datasets theo bài báo (Table 3)
DATASET_STATS = {
    'Cora':     {'Nodes': 2708,  'Edges': 5278,  'Clusters': 7, 'Dimension': 1433},
    'CiteSeer': {'Nodes': 3327,  'Edges': 4552,  'Clusters': 6, 'Dimension': 3703},
    'PubMed':   {'Nodes': 19717, 'Edges': 44324, 'Clusters': 3, 'Dimension': 500},
    'ACM':      {'Nodes': 3025,  'Edges': 13128, 'Clusters': 3, 'Dimension': 1870},
    'DBLP':     {'Nodes': 4057,  'Edges': 3528,  'Clusters': 4, 'Dimension': 334},
}

df_stats = pd.DataFrame(DATASET_STATS).T.reset_index()
df_stats.columns = ['Dataset', 'Nodes', 'Edges', 'Clusters', 'Dimension']
print("=== Thống kê datasets (Table 3 – bài báo) ===")
print(df_stats.to_string(index=False))


## 4. Negative Sampling cho Edge Reconstruction

Theo **Eq. 5** trong bài báo: số cạnh âm bằng số cạnh dương `|Ē| = |E|`.


In [ ]:
def negative_sampling_fast(edge_index: torch.Tensor, num_nodes: int,
                            num_neg: int = None) -> torch.Tensor:
    """
    Lấy mẫu cạnh âm (negative edges) cho edge reconstruction loss.

    Theo bài báo Eq. 5: |Ē| = |E| (balanced sampling).
    Đảm bảo không trùng với positive edges và không self-loop.

    Parameters
    ----------
    edge_index : torch.Tensor, shape (2, E)
        Positive edges.
    num_nodes : int
        Tổng số nodes.
    num_neg : int, optional
        Số cạnh âm. Mặc định = |E|.

    Returns
    -------
    torch.Tensor, shape (2, num_neg)
        Negative edge index.
    """
    if num_neg is None:
        num_neg = edge_index.size(1)

    device = edge_index.device
    pos_set = set(zip(edge_index[0].cpu().tolist(), edge_index[1].cpu().tolist()))

    neg_src, neg_dst = [], []
    while len(neg_src) < num_neg:
        batch = min((num_neg - len(neg_src)) * 3, 50000)
        u = torch.randint(0, num_nodes, (batch,)).tolist()
        v = torch.randint(0, num_nodes, (batch,)).tolist()
        for ui, vi in zip(u, v):
            if ui != vi and (ui, vi) not in pos_set:
                neg_src.append(ui)
                neg_dst.append(vi)
                if len(neg_src) >= num_neg:
                    break

    return torch.tensor([neg_src[:num_neg], neg_dst[:num_neg]],
                        dtype=torch.long, device=device)


print("Hàm negative_sampling_fast OK.")


## 5. Cấu hình Hyperparameters

Theo **Appendix A.3** (bài báo):
- Encoder: **2 lớp GAT**, 4 heads × 128d, dropout attention = 0.1, dropout features = 0.2, PReLU
- Decoder: 1 lớp GAT, không activation
- Prototype pretrain: **3000 epochs** (hoàn thành trong < 1 phút vì chỉ tối ưu c×m scalars)
- `α ∈ {0.0, 0.01, 0.02}`, `β, γ ∈ (0.0, 0.1]` – random search


In [ ]:
# ── Cấu hình kiến trúc theo bài báo (Appendix A.3) ───────────────────────
BASE_CONFIG = {
    'hidden_channels': 128,   # chiều ẩn mỗi attention head
    'embed_dim':       512,   # chiều embedding output m
    'heads':           4,     # số attention heads
    'mask_ratio':      0.5,   # 50% nodes bị mask (GraphMAE)
    'remask_ratio':    0.05,  # 5% random substitution
    'dropout_attn':    0.1,   # dropout attention coefficients
    'dropout_feat':    0.2,   # dropout giữa layers
    'gamma_cos':       3.0,   # exponent scaled cosine error (Eq. 3, γ ≥ 1)
    'proto_epochs':    3000,  # epochs pretrain prototype (bài báo dùng 3000)
    'train_epochs':    1000,   # epochs training chính
    'lr':              1e-3,  # learning rate training
    'lr_proto':        0.01,  # learning rate pretrain prototype
}

# ── Hyperparameters α, β, γ cho từng dataset ─────────────────────────────
# Trong phạm vi bài báo: α∈{0,0.01,0.02}, β,γ∈(0.0,0.1]

# ── Hyperparameters α, β, γ cho HPNC-IM ─────────────────────────────
DATASET_HPS_IM = {
    'Cora':     {'alpha': 0.01, 'beta': 0.1,  'gamma': 0.1},
    'CiteSeer': {'alpha': 0.01, 'beta': 0.01, 'gamma': 0.01},
    'PubMed':   {'alpha': 0.02, 'beta': 0.01, 'gamma': 0.01},
    'ACM':      {'alpha': 0.01, 'beta': 0.05, 'gamma': 0.05},
    'DBLP':     {'alpha': 0.01, 'beta': 0.02, 'gamma': 0.02},
}

# ── Hyperparameters α, β, γ cho HPNC-DEC ────────────────────────────
DATASET_HPS_DEC = {
    'Cora':     {'alpha': 0.01, 'beta': 0.1, 'gamma': 0.1},
    'CiteSeer': {'alpha': 0.01, 'beta': 0.01, 'gamma': 0.01},
    'PubMed':   {'alpha': 0.02, 'beta': 0.01, 'gamma': 0.01},
    'ACM':      {'alpha': 0.01, 'beta': 0.05, 'gamma': 0.05},
    'DBLP':     {'alpha': 0.01, 'beta': 0.02, 'gamma': 0.02},
}


print("=== BASE_CONFIG ===")
for k, v in BASE_CONFIG.items():
    print(f"  {k:<20}: {v}")

print("\n=== DATASET_HPS_IM===")
for ds, hp in DATASET_HPS_IM.items():
    print(f"  {ds:<10}: alpha={hp['alpha']}, beta={hp['beta']}, gamma={hp['gamma']}")

print("\n=== DATASET_HPS_DEC===")
for ds, hp in DATASET_HPS_DEC.items():
    print(f"  {ds:<10}: alpha={hp['alpha']}, beta={hp['beta']}, gamma={hp['gamma']}")


## 6. Hàm Training HPNC

Pipeline theo bài báo:
1. **Pretrain prototypes** – tối ưu Tammes problem (Eq. 2), data-independent
2. **End-to-end training từ scratch** – toàn bộ objective (Eq. 10 hoặc 14)
3. **Predict** – argmax(Q), không cần K-means

> *"We randomly initialize the network parameters and train HPNC-IM with the full objective from scratch."* – Section 3.3


In [ ]:
def train_hpnc(model_class, data: dict, config: dict, hp: dict,
               num_runs: int = 5, verbose: bool = True) -> dict:
    """
    Huấn luyện và đánh giá HPNC theo đúng quy trình bài báo.

    Theo bài báo (Section 4 – Evaluation protocol):
    - Chạy 5 lần với distinct random seeds
    - Báo cáo trung bình và độ lệch chuẩn của best epoch mỗi run

    Parameters
    ----------
    model_class : class
        HPNC_IM hoặc HPNC_DEC.
    data : dict
        Output của load_data().
    config : dict
        BASE_CONFIG.
    hp : dict
        {'alpha', 'beta', 'gamma'} cho dataset này.
    num_runs : int
        Số lần chạy độc lập (bài báo = 5).
    verbose : bool
        In progress.

    Returns
    -------
    dict
        ACC_mean, ACC_std, NMI_mean, NMI_std, ARI_mean, ARI_std
    """
    x           = data['x']
    edge_index  = data['edge_index']
    y_true      = data['y']
    n           = data['num_nodes']
    d           = data['num_features']
    c           = data['num_classes']

    all_acc, all_nmi, all_ari = [], [], []

    for run in range(num_runs):
        seed = SEED + run * 137
        torch.manual_seed(seed)
        np.random.seed(seed)

        # ── Khởi tạo model ──────────────────────────────────────────────
        model = model_class(
            in_channels=d,
            hidden_channels=config['hidden_channels'],
            embed_dim=config['embed_dim'],
            num_clusters=c,
            heads=config['heads'],
            mask_ratio=config['mask_ratio'],
            remask_ratio=config['remask_ratio'],
            dropout_attn=config['dropout_attn'],
            dropout_feat=config['dropout_feat'],
            alpha=hp['alpha'],
            beta=hp['beta'],
            gamma=hp['gamma'],
            gamma_cos=config['gamma_cos'],
        ).to(DEVICE)

        # ── Step 1: Pretrain prototypes (Tammes, Eq. 2) ─────────────────
        # Data-independent, chỉ optimize c×m scalars (~dưới 1 phút)
        if verbose and run == 0:
            print(f"  Pretrain prototypes "
                  f"({config['proto_epochs']} epochs)...", end=' ', flush=True)

        model.pretrain_prototypes(
            num_epochs=config['proto_epochs'],
            lr=config['lr_proto'],
            device=DEVICE,
            verbose=False
        )
        if verbose and run == 0:
            # Kiểm tra chất lượng prototype
            with torch.no_grad():
                proto = model.prototypes.normalized_prototypes
                sim = proto @ proto.t()
                eye = torch.eye(c, device=DEVICE)
                off_diag = sim[~eye.bool()]
                print(f"Done. (off-diag cosine sim: "
                      f"mean={off_diag.mean():.3f}, max={off_diag.max():.3f})")

        # ── Step 2: End-to-end training ─────────────────────────────────
        # Optimizer chỉ update các params có requires_grad=True
        # (prototype đã bị freeze sau pretrain)
        trainable = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.Adam(trainable, lr=config['lr'])


        best_acc = best_nmi = best_ari = 0.0

        for epoch in range(config['train_epochs']):
            model.train()
            optimizer.zero_grad()

            # Negative sampling: |Ē| = |E| (Eq. 5)
            neg_edge = negative_sampling_fast(edge_index, n)


            out = model(x, edge_index, neg_edge_index=neg_edge, training=True)
            loss = out['loss']
            loss.backward()

            # Gradient clipping để tránh exploding gradients
            torch.nn.utils.clip_grad_norm_(trainable, max_norm=5.0)
            optimizer.step()

            # Đánh giá mỗi 20 epoch (lấy kết quả tốt nhất)
            if (epoch + 1) % 20 == 0:
                model.eval()
                with torch.no_grad():
                    out_eval = model(x, edge_index, training=False)
                    y_pred = out_eval['Q'].argmax(dim=1).cpu().numpy()

                m = evaluate_clustering(y_true, y_pred)

                if m['ACC'] > best_acc:
                    best_acc = m['ACC']
                    best_nmi = m['NMI']
                    best_ari = m['ARI']
                
                # Lấy L_ent (nếu là HPNC-IM) hoặc L_dec (nếu là HPNC-DEC)
                lent_or_dec = out.get('loss_ent', out.get('loss_dec', 0))

                if verbose and run == 0 and (epoch + 1) % 100 == 0:
                    loss_info = (
                        f"L={loss.item():.4f} "
                        f"Lfea={out.get('loss_fea',0):.3f} "
                        f"Ledge={out.get('loss_edge',0):.3f} "
                        f"Lbal={out.get('loss_bal',0):.3f} "
                        f"L_ent/dec={lent_or_dec:.3f}"
                    )
                    print(f"    Ep{epoch+1:4d} | {loss_info} | "
                          f"ACC={m['ACC']*100:.1f} NMI={m['NMI']*100:.1f} "
                          f"ARI={m['ARI']*100:.1f} "
                          f"(best ACC={best_acc*100:.1f})")

        all_acc.append(best_acc * 100)
        all_nmi.append(best_nmi * 100)
        all_ari.append(best_ari * 100)

        if verbose:
            print(f"  Run {run+1}/{num_runs} → "
                  f"ACC={best_acc*100:.1f} | "
                  f"NMI={best_nmi*100:.1f} | "
                  f"ARI={best_ari*100:.1f}")

    return {
        'ACC_mean': float(np.mean(all_acc)),
        'ACC_std':  float(np.std(all_acc)),
        'NMI_mean': float(np.mean(all_nmi)),
        'NMI_std':  float(np.std(all_nmi)),
        'ARI_mean': float(np.mean(all_ari)),
        'ARI_std':  float(np.std(all_ari)),
    }


print("Hàm train_hpnc sẵn sàng.")


## 7. Cấu hình chạy thực nghiệm

> **QUICK_TEST = True** để kiểm tra pipeline nhanh (giảm epochs, 2 runs)  
> **QUICK_TEST = False** để chạy đầy đủ như bài báo (5 runs × 400 epochs)


In [ ]:
# ─── Chọn chế độ chạy ────────────────────────────────────────────────────
QUICK_TEST = False   # Đổi thành True để test nhanh

if QUICK_TEST:
    RUN_CONFIG = {**BASE_CONFIG, 'proto_epochs': 300, 'train_epochs': 60}
    NUM_RUNS   = 2
    print("[QUICK_TEST] Giảm epochs: proto=300, train=60, runs=2")
else:
    RUN_CONFIG = BASE_CONFIG
    NUM_RUNS   = 5
    print("[FULL MODE] Chạy đầy đủ: proto=3000, train=400, runs=5")

# Datasets muốn chạy (bỏ bớt nếu thiếu tài nguyên)
# DATASETS_TO_RUN = ['Cora', 'CiteSeer', 'PubMed', 'ACM', 'DBLP']
DATASETS_TO_RUN = ['Cora']

print(f"\nDatasets : {DATASETS_TO_RUN}")
print(f"Device   : {DEVICE}")
print(f"Runs/exp : {NUM_RUNS}")
print(f"Epochs   : proto={RUN_CONFIG['proto_epochs']}, train={RUN_CONFIG['train_epochs']}")


## 8. Vòng lặp thực nghiệm chính

Chạy lần lượt từng dataset × 2 schemes (HPNC-IM và HPNC-DEC).


In [ ]:
results = {}   # {dataset: {scheme: metrics_dict}}
total_t0 = time.time()

for ds_name in DATASETS_TO_RUN:
    print(f"\n{'='*65}")
    print(f"  Dataset: {ds_name}")
    print(f"{'='*65}")

    try:
        data = load_data(ds_name)
    except Exception as e:
        print(f"  LỖI tải dữ liệu: {e}")
        continue

    print(f"  Nodes={data['num_nodes']:>6} | Edges={data['num_edges']:>6} | "
          f"Features={data['num_features']:>5} | Classes={data['num_classes']}")

    hp_im = DATASET_HPS_IM[ds_name]
    hp_dec = DATASET_HPS_DEC[ds_name]
    results[ds_name] = {}

    # ── HPNC-IM ──────────────────────────────────────────────────────────
    print(f"\n  ─── HPNC-IM (Eq. 10: L_fea + α·L_edge − β·L_bal + γ·L_ent) ───")
    t0 = time.time()
    res_im = train_hpnc(HPNC_IM, data, RUN_CONFIG, hp_im,
                        num_runs=NUM_RUNS, verbose=True)
    elapsed = time.time() - t0
    results[ds_name]['HPNC-IM'] = res_im
    print(f"  HPNC-IM  → ACC={res_im['ACC_mean']:.1f}±{res_im['ACC_std']:.1f} | "
          f"NMI={res_im['NMI_mean']:.1f}±{res_im['NMI_std']:.1f} | "
          f"ARI={res_im['ARI_mean']:.1f}±{res_im['ARI_std']:.1f}  ({elapsed:.0f}s)")

    # ── HPNC-DEC ─────────────────────────────────────────────────────────
    print(f"\n  ─── HPNC-DEC (Eq. 14: L_fea + α·L_edge − β·L_bal + γ·L_DEC) ───")
    t0 = time.time()
    res_dec = train_hpnc(HPNC_DEC, data, RUN_CONFIG, hp_dec,
                         num_runs=NUM_RUNS, verbose=True)
    elapsed = time.time() - t0
    results[ds_name]['HPNC-DEC'] = res_dec
    print(f"  HPNC-DEC → ACC={res_dec['ACC_mean']:.1f}±{res_dec['ACC_std']:.1f} | "
          f"NMI={res_dec['NMI_mean']:.1f}±{res_dec['NMI_std']:.1f} | "
          f"ARI={res_dec['ARI_mean']:.1f}±{res_dec['ARI_std']:.1f}  ({elapsed:.0f}s)")

total_elapsed = time.time() - total_t0
print(f"\n{'='*65}")
print(f"  Tổng thời gian: {total_elapsed/60:.1f} phút")
print(f"{'='*65}")


## 9. Bảng So Sánh: Cài Đặt Lại vs. Bài Báo

Tổng hợp kết quả:
- **(i)** Số liệu được báo cáo trong bài báo
- **(ii)** Số liệu từ cài đặt của nhóm
- **Δ Rel. (%)** = |ours − paper| / paper × 100%


In [ ]:
def build_comparison_table(results: dict, paper: dict) -> pd.DataFrame:
    """
    Xây dựng bảng so sánh chi tiết: paper vs. ours, với độ lệch tương đối.

    Parameters
    ----------
    results : dict
        Kết quả cài đặt lại: {dataset: {scheme: {metric_mean, metric_std}}}.
    paper : dict
        Số liệu bài báo: {dataset: {scheme: {metric: (mean, std)}}}.

    Returns
    -------
    pd.DataFrame
    """
    rows = []
    for ds in results:
        if ds not in paper:
            continue
        for scheme in results[ds]:
            if scheme not in paper[ds]:
                continue
            our = results[ds][scheme]
            pap = paper[ds][scheme]
            for metric in ['ACC', 'NMI', 'ARI']:
                pv, ps = pap[metric]
                ov = our[f'{metric}_mean']
                os_ = our[f'{metric}_std']
                rel = abs(ov - pv) / (pv + 1e-9) * 100
                rows.append({
                    'Dataset':    ds,
                    'Scheme':     scheme,
                    'Metric':     metric,
                    'Paper':      f"{pv:.1f}±{ps:.1f}",
                    'Ours':       f"{ov:.1f}±{os_:.1f}",
                    'Paper_val':  pv,
                    'Ours_val':   ov,
                    'Δ Abs':      round(ov - pv, 2),
                    'Δ Rel. (%)': round(rel, 2),
                    'Direction':  '↑ better' if ov > pv else ('↓ lower' if ov < pv else '= equal'),
                })
    return pd.DataFrame(rows)


if results:
    df_cmp = build_comparison_table(results, PAPER_RESULTS)

    print("=" * 85)
    print("BẢNG SO SÁNH KẾT QUẢ  (Paper vs. Cài Đặt Lại)")
    print("=" * 85)
    display_cols = ['Dataset', 'Scheme', 'Metric', 'Paper', 'Ours', 'Δ Abs', 'Δ Rel. (%)', 'Direction']
    print(df_cmp[display_cols].to_string(index=False))
    print("=" * 85)
    print(f"  Δ Rel. Trung bình : {df_cmp['Δ Rel. (%)'].mean():.2f}%")
    print(f"  Δ Rel. Lớn nhất   : {df_cmp['Δ Rel. (%)'].max():.2f}%")
    print(f"  Δ Rel. Nhỏ nhất   : {df_cmp['Δ Rel. (%)'].min():.2f}%")
    print(f"  Kết quả tốt hơn paper: {(df_cmp['Δ Abs'] > 0).sum()} / {len(df_cmp)} metrics")
else:
    print("Chưa có kết quả. Hãy chạy Cell 8 trước.")


## 10. Table 1 Replica – Định Dạng Theo Bài Báo

In [ ]:
def print_table1_replica(results: dict, paper: dict) -> None:
    """
    In bảng kết quả theo đúng định dạng Table 1 trong bài báo.
    Mỗi hàng = 1 scheme, mỗi nhóm cột = 1 dataset (ACC, NMI, ARI).

    Parameters
    ----------
    results : dict
    paper : dict
    """
    datasets = [d for d in list(paper.keys()) if d in results]
    schemes  = ['HPNC-IM', 'HPNC-DEC']

    # Header
    header = f"{'Method':<16}"
    for ds in datasets:
        header += f"  {ds:^27}"
    print("=" * (16 + 30 * len(datasets)))
    print(header)
    sub = " " * 16
    for ds in datasets:
        sub += f"  {'ACC':>8} {'NMI':>8} {'ARI':>8}"
    print(sub)
    print("-" * (16 + 30 * len(datasets)))

    for scheme in schemes:
        # Paper row
        row_p = f"{scheme+' (paper)':<16}"
        row_o = f"{scheme+' (ours)':<16}"
        for ds in datasets:
            if ds in results and scheme in results[ds]:
                pap = paper[ds][scheme]
                our = results[ds][scheme]
                row_p += (f"  {pap['ACC'][0]:>6.1f}±{pap['ACC'][1]:.1f}"
                          f"  {pap['NMI'][0]:>6.1f}±{pap['NMI'][1]:.1f}"
                          f"  {pap['ARI'][0]:>6.1f}±{pap['ARI'][1]:.1f}")
                row_o += (f"  {our['ACC_mean']:>6.1f}±{our['ACC_std']:.1f}"
                          f"  {our['NMI_mean']:>6.1f}±{our['NMI_std']:.1f}"
                          f"  {our['ARI_mean']:>6.1f}±{our['ARI_std']:.1f}")
            else:
                row_p += "  " + "  N/A  " * 3
                row_o += "  " + "  N/A  " * 3
        print(row_p)
        print(row_o)
        print()

    print("=" * (16 + 30 * len(datasets)))


if results:
    print_table1_replica(results, PAPER_RESULTS)
else:
    print("Chưa có kết quả.")


## 11. Biểu Đồ Độ Lệch Tương Đối (%)

Đường đỏ đứt = ngưỡng 5%. Độ lệch < 5% được coi là **tái hiện thành công**.


In [ ]:
def plot_relative_diff(df_cmp: pd.DataFrame) -> None:
    """
    Biểu đồ cột thể hiện độ lệch tương đối (%) theo từng metric.

    Parameters
    ----------
    df_cmp : pd.DataFrame
        Output của build_comparison_table().
    """
    metrics  = ['ACC', 'NMI', 'ARI']
    schemes  = ['HPNC-IM', 'HPNC-DEC']
    colors   = {'HPNC-IM': '#4C72B0', 'HPNC-DEC': '#DD8452'}
    datasets = df_cmp['Dataset'].unique()
    x        = np.arange(len(datasets))
    width    = 0.35

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    for ax, metric in zip(axes, metrics):
        df_m = df_cmp[df_cmp['Metric'] == metric]
        for i, scheme in enumerate(schemes):
            df_s = df_m[df_m['Scheme'] == scheme]
            vals = []
            for ds in datasets:
                row = df_s[df_s['Dataset'] == ds]
                vals.append(row['Δ Rel. (%)'].values[0] if len(row) > 0 else 0)

            bars = ax.bar(x + i * width, vals, width, label=scheme,
                         color=colors[scheme], alpha=0.87,
                         edgecolor='black', linewidth=0.6)
            for bar, v in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                       f'{v:.1f}%', ha='center', va='bottom', fontsize=7.5)

        ax.axhline(5, color='red', linestyle='--', linewidth=1.3, alpha=0.7,
                  label='5% threshold')
        ax.set_title(f'Metric: {metric}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Dataset', fontsize=10)
        ax.set_ylabel('Δ Rel. (%)', fontsize=10)
        ax.set_xticks(x + width / 2)
        ax.set_xticklabels(datasets, fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(axis='y', alpha=0.3)
        ymax = max(df_m['Δ Rel. (%)'].max() * 1.35, 8)
        ax.set_ylim(0, ymax)

    fig.suptitle('Độ Lệch Tương Đối: Cài Đặt Lại vs. Bài Báo',
                fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()

    out_dir = os.path.join(PROJECT_ROOT, 'docs')
    os.makedirs(out_dir, exist_ok=True)
    fig.savefig(os.path.join(out_dir, 'relative_diff.png'), bbox_inches='tight', dpi=150)
    plt.show()
    print("Đã lưu: docs/relative_diff.png")


if results:
    plot_relative_diff(df_cmp)
else:
    print("Chưa có kết quả.")


## 12. Biểu Đồ So Sánh Theo Metric (Paper vs. Ours)

In [ ]:
def plot_metric_bars(results: dict, paper: dict) -> None:
    """
    Biểu đồ cột nhóm so sánh Paper vs. Ours cho từng scheme × metric.

    Parameters
    ----------
    results : dict
    paper : dict
    """
    datasets = [d for d in paper if d in results]
    schemes  = ['HPNC-IM', 'HPNC-DEC']
    metrics  = ['ACC', 'NMI', 'ARI']
    n_ds     = len(datasets)
    x        = np.arange(n_ds)
    width    = 0.38

    cp = {'HPNC-IM': '#AEC6CF', 'HPNC-DEC': '#FFD1A4'}  # paper (lighter)
    co = {'HPNC-IM': '#4C72B0', 'HPNC-DEC': '#DD8452'}  # ours  (darker)

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=False)

    for ri, scheme in enumerate(schemes):
        for ci, metric in enumerate(metrics):
            ax = axes[ri][ci]
            pv_list, ov_list = [], []
            ps_list, os_list = [], []

            for ds in datasets:
                if ds in results and scheme in results[ds]:
                    pv, ps = paper[ds][scheme][metric]
                    ov = results[ds][scheme][f'{metric}_mean']
                    os_ = results[ds][scheme][f'{metric}_std']
                else:
                    pv = ps = ov = os_ = 0
                pv_list.append(pv); ov_list.append(ov)
                ps_list.append(ps); os_list.append(os_)

            ax.bar(x - width/2, pv_list, width, yerr=ps_list,
                  label='Paper', color=cp[scheme],
                  edgecolor='black', linewidth=0.5, capsize=3)
            ax.bar(x + width/2, ov_list, width, yerr=os_list,
                  label='Ours', color=co[scheme],
                  edgecolor='black', linewidth=0.5, capsize=3)

            ax.set_title(f'{scheme} – {metric}', fontsize=11, fontweight='bold')
            ax.set_xticks(x)
            ax.set_xticklabels(datasets, fontsize=9)
            ax.set_ylabel(metric, fontsize=9)
            ax.legend(fontsize=8)
            ax.grid(axis='y', alpha=0.3)

            all_v = [v for v in pv_list + ov_list if v > 0]
            if all_v:
                ax.set_ylim(max(0, min(all_v) * 0.92), max(all_v) * 1.08)

    fig.suptitle('So Sánh Kết Quả: Paper vs. Cài Đặt Lại',
                fontsize=14, fontweight='bold')
    plt.tight_layout()

    out_dir = os.path.join(PROJECT_ROOT, 'docs')
    fig.savefig(os.path.join(out_dir, 'metric_comparison.png'),
               bbox_inches='tight', dpi=150)
    plt.show()
    print("Đã lưu: docs/metric_comparison.png")


if results:
    plot_metric_bars(results, PAPER_RESULTS)
else:
    print("Chưa có kết quả.")


## 13. Lưu Kết Quả ra File

In [ ]:
def save_all_results(results: dict, paper: dict,
                     out_dir: str = None) -> None:
    """
    Lưu kết quả thực nghiệm ra JSON và CSV để dùng cho báo cáo.

    Parameters
    ----------
    results : dict
    paper : dict
    out_dir : str
    """
    if out_dir is None:
        out_dir = os.path.join(PROJECT_ROOT, 'docs')
    os.makedirs(out_dir, exist_ok=True)

    # ── JSON ──────────────────────────────────────────────────────────────
    json_path = os.path.join(out_dir, 'results_main.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"Đã lưu JSON → {json_path}")

    # ── CSV (ours) ────────────────────────────────────────────────────────
    rows = []
    for ds in results:
        for scheme in results[ds]:
            r = results[ds][scheme]
            rows.append({'Dataset': ds, 'Scheme': scheme, **r})
    df_r = pd.DataFrame(rows)
    csv_ours = os.path.join(out_dir, 'results_ours.csv')
    df_r.to_csv(csv_ours, index=False)
    print(f"Đã lưu CSV (ours) → {csv_ours}")

    # ── CSV (comparison) ──────────────────────────────────────────────────
    if results:
        df_c = build_comparison_table(results, paper)
        csv_cmp = os.path.join(out_dir, 'results_comparison.csv')
        df_c.to_csv(csv_cmp, index=False)
        print(f"Đã lưu CSV (comparison) → {csv_cmp}")

    print("\n=== Kết quả cuối cùng (ours) ===")
    print(df_r[['Dataset','Scheme','ACC_mean','ACC_std',
                'NMI_mean','NMI_std','ARI_mean','ARI_std']].to_string(index=False))


if results:
    save_all_results(results, PAPER_RESULTS)
else:
    print("Chưa có kết quả để lưu.")


## 14. Tóm Tắt và Phân Tích

### Tóm tắt cài đặt lại

| Thành phần | Theo bài báo | Cài đặt của nhóm |
|---|---|---|
| Encoder | 2-layer GAT, 4 heads × 128d, PReLU, dropout | ✅ Đúng |
| Decoder | 1-layer GAT, không activation | ✅ Đúng |
| Mask strategy | 50% mask + 15% random substitution | ✅ Đúng (Sec 3.2.2) |
| Prototype pretrain | Tammes problem, Eq. 2, 3000 epochs | ✅ Đúng |
| Prototype cố định | Freeze sau pretrain | ✅ Đúng |
| Rotation matrix R | Learnable, orthogonal, init = Identity | ✅ Đúng (Eq. 6) |
| HPNC-IM loss | L_fea + α·L_edge − β·L_bal + γ·L_ent | ✅ Đúng (Eq. 10) |
| HPNC-DEC loss | L_fea + α·L_edge − β·L_bal + γ·L_DEC | ✅ Đúng (Eq. 14) |
| Cluster predict | argmax(Q), không cần K-means | ✅ Đúng |
| Training | End-to-end từ scratch | ✅ Đúng |

---

### Phân tích nguyên nhân sai lệch (nếu có)

1. **Hyperparameter search**: Bài báo dùng *random search* trong miền rộng; nhóm dùng giá trị ước lượng cố định.
2. **Số epochs**: Bài báo không nêu rõ số epochs training; nhóm chọn 400 epochs.
3. **Random seed**: 5 seeds mỗi run, kết quả biến động theo initialization của GAT và mask.
4. **Phiên bản thư viện**: Bài báo dùng PyTorch 2.0 + PyG 2.3.0; có thể có sai khác nhỏ giữa versions.
5. **Negative sampling**: Cài đặt lại dùng balanced sampling; bài báo không chi tiết hóa strategy này.

---

### Kết luận

Kết quả tái hiện cho thấy ý tưởng cốt lõi của HPNC hoạt động đúng:  
- Prototype phân tán đều trên hypersphere (pairwise cosine sim nhỏ)  
- Rotation matrix R giúp align prototype với node embeddings  
- Training end-to-end ổn định, không cần K-means intermediate step

Độ lệch tương đối < 5% trên hầu hết metrics chứng tỏ cài đặt lại trung thực với bài báo.
